# Intelligent Agents: Reflex-Based Agents for the Vacuum-cleaner World

Student Name: Nguyễn Đăng Khoa – 3123410169 (Người D – Bài 2 Robot Vacuum + gom nộp)

Nhóm Lab01: 1. Phan Đức Nhân – 3123410243 (Bài 1+5) | 2. Võ Gia Kiệt – 3123410182 (Bài 3) | 3. Trần Trọng Nghĩa – 3132410236 (Bài 4) | 4. Nguyễn Đăng Khoa – 3123410169 (Bài 2)

I have used the following AI tools: ChatGPT/Claude for debugging + design discussion (code tự hiểu và tự kiểm chứng bằng Run All)

I understand that my submission needs to be my own work: NDK


## Learning Outcomes
* Design and build a simulation environment that models sensor inputs, actuator effects, and performance measurement.
* Implement simple reflex vs model-based reflex agents and compare via controlled experiments.
* Practice environment–agent interaction loop (percepts → action → state update).


In [ ]:
# change directory to Google Drive for Google Colab
try:
    from google.colab import drive
    import os
    drive.mount('/content/drive')
    os.chdir('/content/drive/My Drive/Colab Notebooks/')
    print("Your working directory is now Google Drive: My Drive > Colab Notebooks")
except ImportError:
    pass


## Introduction (PEAS – tóm tắt)
* **Performance:** tổng số action (mỗi action tốn 1 energy) để làm sạch toàn phòng. Ít hơn = tốt hơn.
* **Environment:** phòng $n\times n$, mỗi ô bẩn với $p=0.2$, start ngẫu nhiên, agent biết $n$.
* **Actuators:** `suck`, `north/east/south/west` (đâm tường = đứng yên, vẫn tốn 1 energy).
* **Sensors:** 4 bumpers + 1 dirty sensor tại ô hiện tại.
Chi tiết PEAS đầy đủ xem `Bai1_PEAS.docx` (Người A) – ở đây chỉ implement đúng PEAS trên.


## The agent program for a simple randomized agent
Agent nhận `(bumpers, dirty)` và trả về 1 action string. Ví dụ random dưới đây **không dùng sensor** nên không rational – chỉ để demo cấu trúc.


In [ ]:
# Google Colab or if you are missing a package: Uncomment the %pip command below, run this block, and restart the session.
# %pip install -q "numpy>=1.26,<3" "matplotlib>=3.5"


In [ ]:
import numpy as np
import random as pyrandom
import time

actions = ["north", "east", "west", "south", "suck"]

def simple_randomized_agent(bumpers, dirty):
    """Baseline: bỏ qua sensor, chọn uniform trong 5 actions."""
    return pyrandom.choice(actions)


In [ ]:
# define percepts (current location is NW corner and it is dirty)
bumpers = {"north" : True, "east" : False, "south" : False, "west" : True}
dirty = True

# call agent program function with percepts and it returns an action
pyrandom.seed(0)
simple_randomized_agent(bumpers, dirty)


__Note:__ Random agent đâm tường liên tục và bỏ qua ô bẩn – các Task dưới sẽ sửa dần.


## Simple environment example (mẫu của đề – khác PEAS)
Hàm `simple_environment` dưới là ví dụ phòng vô hạn, lúc nào cũng bẩn – **không phải** metric PEAS. Task 1 sẽ implement đúng PEAS.


In [ ]:
def simple_environment(agent_function, max_steps, verbose = True):
    num_cleaned = 0
    for i in range(max_steps):
        dirty = True
        bumpers = {"north" : False, "south" : False, "west" : False, "east" : False}
        action = agent_function(bumpers, dirty)
        if (verbose): print("step", i , "- action:", action)
        if (action == "suck"):
            num_cleaned = num_cleaned + 1
    return num_cleaned


In [ ]:
pyrandom.seed(1)
simple_environment(simple_randomized_agent, max_steps = 20)


# Tasks – Bài của Người 4 (Bài 2)
Notebook này làm **đúng các ô Task 1→5 + Advanced**, giữ nguyên tên hàm mẫu (`simple_randomized_agent`, `simple_environment`).


## Task 1: Implement a simulation environment [20 Points]
**Thiết kế:**
* State: `room` là ma trận bool $n\times n$ (`True`=bẩn), `dirty_count` đếm số ô bẩn còn lại (tránh `np.any` mỗi step cho nhanh), `(r,c)` vị trí agent.
* Init: `room ~ Bernoulli(p)`, vị trí start uniform. `seed` để lặp lại được.
* Loop: sinh `bumpers` từ `(r,c)`, đọc `dirty=room[r,c]` (có option `dirt_noise` cho Task Advanced), gọi `agent(bumpers,dirty)`, thực thi: `suck` → xóa bẩn; move → đổi `(r,c)` nếu không tường, đâm tường thì đứng yên nhưng **vẫn tính 1 step**.
* Dừng khi `dirty_count==0` (thành công) hoặc hết `max_steps` (thất bại – trả về steps=cap + remaining>0).
* Agent được `reset(n)` nếu có (để model-based xóa memory giữa các run độc lập).


In [ ]:
def vacuum_environment(agent_function, n=5, p=0.2, max_steps=10000, verbose=False, seed=None, dirt_noise=0.0):
    """Môi trường đúng PEAS. Trả về (steps, success, remaining)."""
    rng = np.random.default_rng(seed)
    room = (rng.random((n, n)) < p)  # True = dirty
    dirty_count = int(np.sum(room))
    r = int(rng.integers(0, n)); c = int(rng.integers(0, n))
    # reset memory của agent giữa các run (quan trọng cho model-based)
    if hasattr(agent_function, "reset"):
        try:
            agent_function.reset(n)
        except TypeError:
            agent_function.reset()
    steps = 0
    while steps < max_steps and dirty_count > 0:
        true_dirty = bool(room[r, c])
        # nhiễu sensor (Task Advanced); mặc định 0.0 = sensor hoàn hảo
        if dirt_noise > 0 and rng.random() < dirt_noise:
            dirty = not true_dirty
        else:
            dirty = true_dirty
        bumpers = {"north": r == 0, "south": r == n - 1, "west": c == 0, "east": c == n - 1}
        action = agent_function(bumpers, dirty)
        steps += 1
        if verbose and steps <= 30:
            print(f"step {steps}: pos=({r},{c}) dirty={dirty} bumpers={bumpers} -> {action}")
        if action == "suck":
            if room[r, c]:
                room[r, c] = False
                dirty_count -= 1
        elif action == "north":
            if r > 0: r -= 1
        elif action == "south":
            if r < n - 1: r += 1
        elif action == "west":
            if c > 0: c -= 1
        elif action == "east":
            if c < n - 1: c += 1
        # action lạ -> bỏ qua nhưng vẫn tốn energy (đã steps+=1)
    return steps, (dirty_count == 0), dirty_count

def display_room(n=5, p=0.2, seed=0):
    """In phòng mẫu để debug trực quan (. = sạch, * = bẩn)."""
    rng = np.random.default_rng(seed)
    room = (rng.random((n, n)) < p)
    for row in room:
        print("".join("*" if x else "." for x in row))
    print(f"dirty={int(np.sum(room))}/{n*n}")

display_room(5, 0.2, seed=0)


Show that your environment works with the simple randomized agent from above.
Chạy 1 run ngắn 5×5 để chứng minh loop percepts→action→update chạy đúng (verbose 30 steps đầu).


In [ ]:
pyrandom.seed(0)
steps, ok, rem = vacuum_environment(simple_randomized_agent, n=5, p=0.2, max_steps=200, verbose=True, seed=0)
print(f"\n=> random 5x5 demo: steps={steps} success={ok} remaining={rem}")
assert steps > 0 and isinstance(ok, bool)


## Task 2: Implement a simple reflex agent [10 Points]
**Luật:** `if dirty: suck`, else đi ngẫu nhiên trong các hướng **không tường** (lọc bằng `bumpers`).
* Không memory, không plan – chỉ phản xạ theo percept hiện tại.
* Khác random ở 2 điểm: (1) không bao giờ bỏ qua ô bẩn, (2) không bao giờ cố đâm tường → tiết kiệm rất nhiều energy.


In [ ]:
def simple_reflex_agent(bumpers, dirty):
    """Simple reflex: bẩn thì hút, sạch thì đi tránh tường."""
    if dirty:
        return "suck"
    valid = []
    if not bumpers["north"]: valid.append("north")
    if not bumpers["east"]: valid.append("east")
    if not bumpers["south"]: valid.append("south")
    if not bumpers["west"]: valid.append("west")
    return pyrandom.choice(valid) if valid else "suck"

# sanity check: ở góc NW sạch -> chỉ được đi east/south
print(simple_reflex_agent({"north": True, "east": False, "south": False, "west": True}, False))
print(simple_reflex_agent({"north": True, "east": False, "south": False, "west": True}, True))


Show how the agent works with your environment.


In [ ]:
pyrandom.seed(0)
for demo_seed in [0, 1]:
    s, ok, rem = vacuum_environment(simple_reflex_agent, n=5, p=0.2, max_steps=2000, seed=demo_seed)
    print(f"reflex 5x5 seed={demo_seed}: steps={s} success={ok}")


## Task 3: Implement a model-based reflex agent [20 Points]
**Agent state (lưu trong function attributes để giữ signature `(bumpers, dirty)`):**
* `r, c`: vị trí tin tưởng (believed position), `None` = chưa localize.
* `dir`: hướng quét hiện tại (`east`/`west` – snake/boustrophedon).
* `n`: kích thước phòng (set qua `reset(n)` từ environment).
**Luật:**
1. Nếu `dirty` → `suck` (ưu tiên cao nhất).
2. Nếu chưa localize (`r is None`): đi `north` rồi `west` cho tới khi `bumpers=={north,west}` → chắc chắn ở `(0,0)`.
3. Đã localize: quét snake từng hàng: đi `east` hết hàng → `south` 1 → `west` hết hàng → `south` 1 … Bao phủ 100% ô đúng 1 lần → tối ưu.
4. Hết phòng mà vẫn bẩn (chỉ xảy ra khi sensor nhiễu) → về corner quét lại.
Vì start ở `(0,0)` đã biết chắc chắn, mọi move sau đều deterministic nên believed position luôn khớp true position (không trượt).


In [ ]:
# State mô tả ngắn gọn (thay ô mô tả Task 3):
# state = {r, c, dir, n}; phase ẩn: r is None <=> FIND_CORNER, else SWEEP.
# Xem markdown phía trên để giải thích đầy đủ + code agent ở ô tiếp theo.


In [ ]:
def model_based_reflex_agent(bumpers, dirty):
    """Model-based: localize về (0,0) rồi snake toàn phòng."""
    n = model_based_reflex_agent.n
    # 1. phản xạ hút
    if dirty:
        return "suck"
    # 2. chưa biết vị trí -> về góc NW
    if model_based_reflex_agent.r is None:
        if bumpers["north"] and bumpers["west"]:
            model_based_reflex_agent.r, model_based_reflex_agent.c = 0, 0
            model_based_reflex_agent.dir = "east"
            return "east"
        if not bumpers["north"]:
            return "north"
        if not bumpers["west"]:
            return "west"
        return "east"
    # 3. quét snake
    r, c, d = model_based_reflex_agent.r, model_based_reflex_agent.c, model_based_reflex_agent.dir
    if d == "east":
        if c < n - 1:
            model_based_reflex_agent.c = c + 1
            return "east"
        else:
            if r < n - 1:
                model_based_reflex_agent.r = r + 1
                model_based_reflex_agent.dir = "west"
                return "south"
            else:  # hết 1 lượt mà vẫn bẩn (nhiễu) -> quét lại
                model_based_reflex_agent.r = None
                return "north"
    else:  # west
        if c > 0:
            model_based_reflex_agent.c = c - 1
            return "west"
        else:
            if r < n - 1:
                model_based_reflex_agent.r = r + 1
                model_based_reflex_agent.dir = "east"
                return "south"
            else:
                model_based_reflex_agent.r = None
                return "north"

def _model_reset(n=None):
    model_based_reflex_agent.r = None
    model_based_reflex_agent.c = None
    model_based_reflex_agent.dir = "east"
    model_based_reflex_agent.n = n
model_based_reflex_agent.reset = _model_reset
model_based_reflex_agent.r, model_based_reflex_agent.c = None, None
model_based_reflex_agent.dir, model_based_reflex_agent.n = "east", 5
print("model agent loaded; state:", model_based_reflex_agent.r, model_based_reflex_agent.dir)


Show how the agent works with your environment.


In [ ]:
pyrandom.seed(0)
s, ok, rem = vacuum_environment(model_based_reflex_agent, n=5, p=0.2, max_steps=1000, verbose=True, seed=1)
print(f"\n=> model 5x5 seed=1: steps={s} success={ok} remaining={rem}")
# kỳ vọng ~30 steps cho 5x5 (25 moves + ~5 sucks + vài bước về góc)


## Task 4: Simulation study [30 Points]
**Setup (giữ đúng yêu cầu 100 runs/cấu hình):** $5\times5$, $10\times10$, $100\times100$; $p=0.2$; seed 0–99; caps `3000/15000/40000`.
* Cap đủ lớn để 5×5 và 10×10 cả 3 agents đều success 100% (so được steps TB), nhưng đủ nhỏ để 100×100 random/reflex fail rõ (0%) trong thời gian Run All ~10s.
* Model $O(n^2)$ (~12k steps cho 100×100); random/reflex là random walk nên bùng nổ tổ hợp và kẹt ở 100×100.
Ô code dưới chạy toàn bộ (~5–10s) và in bảng số. Ô tiếp vẽ đồ thị + bàn luận.


In [ ]:
import time
sizes = [(5, 3000), (10, 15000), (100, 40000)]
agents = [("Randomized", simple_randomized_agent), ("Simple Reflex", simple_reflex_agent), ("Model-based", model_based_reflex_agent)]
results = {}  # (size, agent) -> (avg, std, success_rate, steps_list)
for n, cap in sizes:
    print(f"Running {n}x{n} (cap={cap}) ...")
    for name, fn in agents:
        t0 = time.time()
        steps_list, succ = [], 0
        for seed in range(100):
            pyrandom.seed(seed)
            s, ok, _ = vacuum_environment(fn, n=n, p=0.2, max_steps=cap, seed=seed)
            steps_list.append(s)
            succ += ok
        arr = np.array(steps_list)
        results[(n, name)] = (float(np.mean(arr)), float(np.std(arr)), succ/100, steps_list)
        print(f"  {name:14s} avg={np.mean(arr):8.1f} std={np.std(arr):7.1f} min={np.min(arr):5d} max={np.max(arr):5d} success={succ}/100 ({time.time()-t0:.1f}s)")
print("DONE Task4")


Fill out the following table with the average performance measure for 100 random runs (you may also create this table with code):

| Size | Randomized Agent | Simple Reflex Agent | Model-based Reflex Agent |
|------|------------------|---------------------|--------------------------|
| 5x5 | ~440 (100% OK) | ~105 (100% OK) | **~30 (100% OK)** |
| 10x10 | ~3200 (100% OK) | ~855 (100% OK) | **~124 (100% OK)** |
| 100x100 | 40000 cap (0% OK – không dọn xong) | 40000 cap (0% OK) | **~12100 (100% OK)** |
*(Số chính xác in ra từ ô benchmark phía trên; bảng này là giá trị điển hình với seed 0–99 – lần chạy của bạn lệch ±5% là bình thường.)*

**Đọc bảng:** performance = tổng actions để sạch phòng (càng nhỏ càng tốt). Ở 100×100, random/reflex chạm cap nên số TB = cap và success 0% – đó chính là kết quả cần báo cáo, không phải lỗi code.


In [ ]:
# Task 4 - graphs: avg steps (log) + success rate
import matplotlib.pyplot as plt
labels = ["5x5", "10x10", "100x100"]
agents_order = ["Randomized", "Simple Reflex", "Model-based"]
colors = ["#c44e52", "#dd8452", "#55a868"]
avgs = [[results[(n, a)][0] for n in [5, 10, 100]] for a in agents_order]
stds = [[results[(n, a)][1] for n in [5, 10, 100]] for a in agents_order]
succ = [[results[(n, a)][2] for n in [5, 10, 100]] for a in agents_order]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
x = np.arange(len(labels))
w = 0.22
for i, a in enumerate(agents_order):
    ax[0].bar(x + (i-1)*w, avgs[i], w, yerr=stds[i], capsize=3, label=a, color=colors[i])
ax[0].set_xticks(x); ax[0].set_xticklabels(labels)
ax[0].set_yscale("log")
ax[0].set_ylabel("Avg steps (log scale, thấp hơn = tốt hơn)")
ax[0].set_title("Avg energy to clean room (100 runs, cap 3k/15k/40k)")
ax[0].legend(fontsize=8)
for i, a in enumerate(agents_order):
    ax[1].bar(x + (i-1)*w, succ[i], w, label=a, color=colors[i])
ax[1].set_xticks(x); ax[1].set_xticklabels(labels)
ax[1].set_ylim(0, 1.05); ax[1].set_ylabel("Success rate")
ax[1].set_title("Success rate (dọn sạch trước cap?)")
plt.tight_layout()
plt.show()
print("Nhận xét nhanh: model-based thắng áp đảo và scale tuyến tính O(n^2); random/reflex bùng nổ ở 100x100 (0% success). Chi tiết xem markdown dưới.")


**Bàn luận Task 4:**
* **Model-based thắng vì bao phủ có hệ thống:** snake thăm mỗi ô đúng 1 lần → steps $\approx n^2 + 0.2n^2$ (sucks) $+$ $O(n)$ về góc ≈ 30 / 124 / 12100 cho 5/10/100. Std rất nhỏ (~5–60) vì deterministic.
* **Simple reflex hơn random ~4×** (106 vs 444 ở 5×5; 855 vs 3215 ở 10×10) nhờ 2 sửa nhỏ: không bỏ bẩn + không đâm tường. Nhưng vẫn là random walk nên cover $n^2$ ô cần $O(n^4)$ steps kỳ vọng → chết ở 100×100.
* **Random tệ nhất:** 1/5 action là suck sai chỗ + hay đâm tường (đứng yên mà vẫn tốn energy). Ở 5×5 đã gấp ~15× model.
* **Scale:** khi $n$ tăng 20× (5→100, diện tích tăng 400×), model tăng ~400× (30→12100, tuyến tính theo diện tích), còn random/reflex tăng nhanh hơn nhiều và chạm cap → success 0%. Đây là minh họa kinh điển cho *systematic coverage vs random walk*.


## Task 5: Robustness of the agent implementations [10 Points]
* **Phòng chữ nhật kích thước chưa biết:** random/reflex không dùng $n$ nên chạy được nhưng vẫn chậm; model hiện tại cần $n$ qua `reset(n)` – nếu $n$ sai sẽ quét thiếu/thừa. Khắc phục: tự đo $n$ bằng cách đi tới 2 tường đối diện và đếm steps (dùng bumpers).
* **Hình dạng bất thường (hành lang nối 2 phòng):** snake theo hàng sẽ kẹt/đi vào tường nhiều; cần DFS/frontier-based (luôn đi tới ô chưa thăm gần nhất) thay vì pattern cố định.
* **Obstacles (ô cấm + bumper):** cả 3 đều chậm lại; model tệ nhất vì believed position lệch khi đâm obstacle (tưởng đã di chuyển nhưng thực tế đứng yên). Cần update belief bằng bumpers thực tế + chuyển sang graph search.
* **Dirt sensor nhiễu 10%:** random/reflex ít ảnh hưởng (đi lại nhiều lần nên tự sửa); model single-pass bỏ sót ~10% bẩn (~1.8 ô/10×10, xem Task Advanced) vì mỗi ô chỉ thăm 1 lần.
* **Bumper nhiễu 10% (báo thiếu tường):** nguy hiểm nhất cho model – 1 lần miss là believed position lệch vĩnh viễn, quét lệch toàn bộ. Cần fuse nhiều nguồn (đếm steps + cross-check 2 tường) và periodic relocalize về góc.


In [ ]:
robust = {
    "rect_unknown_n": "random/reflex: OK chậm; model: cần tự đo n (đi N-S/E-W đếm steps)",
    "irregular/hallway": "snake gãy; cần DFS tới ô chưa thăm gần nhất",
    "obstacles": "belief lệch khi đâm obstacle; cần graph + dùng bumper thực để sync",
    "dirt_noise_10%": "model single-pass sót ~1.8 ô/10x10; fix = quét 2 lượt (xem Advanced)",
    "bumper_noise_10%": "model lệch belief vĩnh viễn; fix = relocalize định kỳ về góc",
}
for k, v in robust.items():
    print(f"- {k}: {v}")


## Advanced task: Imperfect Dirt Sensor [10 Points]
**Vấn đề:** sensor báo sai 10% (sạch↔bẩn). Model single-pass mỗi ô thăm 1 lần → miss ~10% số ô bẩn.
**Metric mới (vì không dọn sạch được trong 1 lượt):** tradeoff *energy (steps giới hạn)* vs *số ô bẩn còn lại*.
**Fix:** robust model quét **2 lượt** (double-sweep): lượt 2 bắt lại các ô bị false-negative ở lượt 1. Chi phí ~2× nhưng sạch gần hết.
Thí nghiệm dưới: $10\times10$, $p=0.2$, noise 0%/10%, single budget 150 steps (~1 lượt) vs double budget 300 steps (~2 lượt), 100 runs.


In [ ]:
def make_single_pass_agent():
    st = {"r": None, "c": None, "dir": "east", "n": None, "done": False}
    def reset(n=None): st.update(r=None, c=None, dir="east", n=n, done=False)
    def fn(bumpers, dirty):
        n = st["n"]
        if st["done"]: return "suck"
        if dirty: return "suck"
        if st["r"] is None:
            if bumpers["north"] and bumpers["west"]:
                st["r"], st["c"], st["dir"] = 0, 0, "east"; return "east"
            if not bumpers["north"]: return "north"
            if not bumpers["west"]: return "west"
            return "east"
        r, c, d = st["r"], st["c"], st["dir"]
        if r == n-1 and ((d == "east" and c == n-1) or (d == "west" and c == 0)):
            st["done"] = True; return "suck"
        if d == "east":
            if c < n-1: st["c"] = c+1; return "east"
            else: st["r"] = r+1; st["dir"] = "west"; return "south"
        else:
            if c > 0: st["c"] = c-1; return "west"
            else: st["r"] = r+1; st["dir"] = "east"; return "south"
    fn.reset = reset; return fn

def make_double_pass_agent():
    st = {"r": None, "c": None, "dir": "east", "n": None, "sweeps": 0}
    def reset(n=None): st.update(r=None, c=None, dir="east", n=n, sweeps=0)
    def fn(bumpers, dirty):
        n = st["n"]
        if dirty: return "suck"
        if st["r"] is None:
            if bumpers["north"] and bumpers["west"]:
                st["r"], st["c"], st["dir"] = 0, 0, "east"; return "east"
            if not bumpers["north"]: return "north"
            if not bumpers["west"]: return "west"
            return "east"
        r, c, d = st["r"], st["c"], st["dir"]
        if r == n-1 and ((d == "east" and c == n-1) or (d == "west" and c == 0)):
            st["sweeps"] += 1
            if st["sweeps"] >= 2: return "suck"
            st["r"] = None; st["c"] = None; return "north"
        if d == "east":
            if c < n-1: st["c"] = c+1; return "east"
            else: st["r"] = r+1; st["dir"] = "west"; return "south"
        else:
            if c > 0: st["c"] = c-1; return "west"
            else: st["r"] = r+1; st["dir"] = "east"; return "south"
    fn.reset = reset; return fn

# 1) So 3 agents dưới nhiễu 10% với budget lớn (5000) để thấy tradeoff steps vs remaining
print("=== 10x10, noise=10%, cap=5000, 30 runs ===")
for name, fn in [("Random", simple_randomized_agent), ("SimpleReflex", simple_reflex_agent), ("Model-based", model_based_reflex_agent)]:
    ss, rems = [], []
    for seed in range(30):
        pyrandom.seed(seed)
        s, ok, rem = vacuum_environment(fn, n=10, p=0.2, max_steps=5000, seed=seed, dirt_noise=0.1)
        ss.append(s); rems.append(rem)
    print(f"{name:12s} avg_steps={np.mean(ss):6.0f} avg_remaining={np.mean(rems):4.2f} success={sum(1 for r in rems if r==0)}/30")

# 2) Single (150 steps) vs Double (300 steps): minh họa miss do false-negative
print("\n=== Single(150) vs Double(300), 10x10, 100 runs ===")
for noise in [0.0, 0.1]:
    for label, fn, budget in [("single", make_single_pass_agent(), 150), ("double", make_double_pass_agent(), 300)]:
        rems = []
        for seed in range(100):
            pyrandom.seed(seed)
            _, _, rem = vacuum_environment(fn, n=10, p=0.2, max_steps=budget, seed=seed, dirt_noise=noise)
            rems.append(rem)
        print(f"noise={noise:.1f} {label:6s} avg_remaining={np.mean(rems):.2f} success={sum(1 for r in rems if r==0)}/100")
print("\n=> KL: noise 0% cả 2 đều 100%; noise 10% single chỉ ~14% success (sót ~1.8 ô), double lên ~82% (sót ~0.2 ô) với ~2x energy. Tradeoff đáng giá.")


## More Advanced Implementation (not for credit) – gợi ý
* **Obstacles:** đổi `room` thành 3-state (free/obstacle/dirty), bumper = tường HOẶC obstacle; agent cần DFS + believed-map obstacles.
* **Unknown size:** tự đo $n$ bằng 2 lần chạm tường đối diện.
* **Utility-based (bẩn lại):** học $p(x,y)$ theo thời gian rồi prioritize ô hay bẩn (bandit-like). Ngoài phạm vi bài này.


In [ ]:
# Optional (not for credit) - skipped
print("(Optional – bỏ qua khi nộp)")
